# EDA on Spotify Tracks dataset

#### Agenda
- Data Processing (and/or Data Wrangling)
- Data Visualisation
- Feature Engineering
- PCA (and/or Descriptive Statistics and/or Correlation)
- Data Clustering

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("dataset.csv")

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.iloc[100:110]

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
# Drop rows with any missing values
df = df.dropna()
print(df.shape)

In [ ]:
# Check for duplicate rows
print('Duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates()
print('Shape after dropping duplicates:', df.shape)

# Data Visualisation

In [ ]:
import plotly.express as px

fig = px.histogram(df, x='popularity', nbins=20,
                   title='Distribution of Track Popularity',
                   color_discrete_sequence=['blue'])
fig.show()

In [ ]:
fig = px.histogram(df, x='danceability', nbins=20,
                   title='Distribution of Danceability',
                   color_discrete_sequence=['green'])
fig.show()

In [ ]:
fig = px.histogram(df, x='energy', nbins=20,
                   title='Distribution of Energy',
                   color_discrete_sequence=['tomato'])
fig.show()

In [ ]:
fig = px.histogram(df, x='valence', nbins=20,
                   title='Distribution of Valence (Musical Positiveness)',
                   color_discrete_sequence=['purple'])
fig.show()

In [ ]:
import matplotlib.pyplot as plt

# Distributions of all the numerical audio features
audio_features = ['danceability', 'energy', 'loudness', 'speechiness',
                  'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))

for ax, feature in zip(axes.flat, audio_features):
    ax.hist(df[feature], bins=30, edgecolor='white', linewidth=0.4)
    ax.set_title(feature)
    ax.set_xlabel(feature)
    ax.set_ylabel('Frequency')

plt.suptitle('Audio Feature Distributions', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
numerical_cols = ['popularity', 'danceability', 'energy', 'loudness', 'speechiness',
                  'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms']

corr = df[numerical_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im)
ax.set_xticks(range(len(numerical_cols)))
ax.set_yticks(range(len(numerical_cols)))
ax.set_xticklabels(numerical_cols, rotation=45, ha='right')
ax.set_yticklabels(numerical_cols)
for i in range(len(numerical_cols)):
    for j in range(len(numerical_cols)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=7)
ax.set_title('Correlation Matrix of Audio Features', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: popularity vs energy (size = loudness)
fig = plt.figure(figsize=(16, 12))

plt.scatter(x=df['energy'],
            y=df['popularity'],
            s=df['loudness'].abs() * 2,
            alpha=0.4,
            edgecolors='w')

plt.xlabel('Energy')
plt.ylabel('Popularity')
plt.title('Track Popularity vs Energy (size = loudness)', y=1.05)

In [ ]:
# Scatter: danceability vs valence (size = energy)
fig = plt.figure(figsize=(16, 12))

plt.scatter(x=df['danceability'],
            y=df['valence'],
            s=df['energy'] * 100,
            alpha=0.4,
            edgecolors='w')

plt.xlabel('Danceability')
plt.ylabel('Valence')
plt.title('Danceability vs Valence (size = energy)', y=1.05)

In [ ]:
# Scatter: acousticness vs energy (size = loudness)
fig = plt.figure(figsize=(16, 12))

plt.scatter(x=df['acousticness'],
            y=df['energy'],
            s=df['loudness'].abs() * 2,
            alpha=0.4,
            edgecolors='w')

plt.xlabel('Acousticness')
plt.ylabel('Energy')
plt.title('Acousticness vs Energy (size = loudness)', y=1.05)

# Feature Engineering

In [ ]:
df.columns

In [ ]:
# Add popularity label based on popularity score
df['popularity_label'] = pd.cut(df['popularity'],
                                 bins=[0, 33, 66, 100],
                                 labels=['low', 'medium', 'high'])

print(df['popularity_label'].value_counts())

In [ ]:
# Add energy category
df['energy_category'] = pd.cut(df['energy'],
                                bins=[0, 0.4, 0.7, 1.0],
                                labels=['low energy', 'medium energy', 'high energy'])

print(df['energy_category'].value_counts())

In [ ]:
# Mood score: weighted combination of valence, danceability and energy
df['mood_score'] = 0.5 * df['valence'] + 0.3 * df['danceability'] + 0.2 * df['energy']

print(df['mood_score'].describe())

In [ ]:
# Duration in minutes (more interpretable than milliseconds)
df['duration_min'] = df['duration_ms'] / 60000

print(df['duration_min'].describe())

In [ ]:
df.columns

In [ ]:
# Popularity distribution by energy category
fig = px.histogram(df, x='popularity', color='energy_category', nbins=20,
                   barmode='overlay',
                   title='Popularity Distribution by Energy Category',
                   color_discrete_sequence=['blue', 'tomato', 'green'])
fig.show()

In [ ]:
# Mood score distribution by popularity label
fig = px.histogram(df, x='mood_score', color='popularity_label', nbins=20,
                   barmode='overlay',
                   title='Mood Score Distribution by Popularity Label',
                   color_discrete_sequence=['blue', 'orange', 'tomato'])
fig.show()

In [ ]:
# Box plot: popularity by energy category
fig = px.box(df, x='energy_category', y='popularity',
             title='Popularity by Energy Category',
             color='energy_category')
fig.show()

In [ ]:
# Box plot: danceability by popularity label
fig = px.box(df, x='popularity_label', y='danceability',
             title='Danceability by Popularity Label',
             color='popularity_label')
fig.show()

# PCA

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd

features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence',
            'tempo', 'duration_min', 'popularity', 'mood_score']

X = df[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=len(features))
X_pca = pca.fit_transform(X_scaled)

print('Explained variance ratio:')
print(pca.explained_variance_ratio_.round(3))
print('\nCumulative:')
print(np.cumsum(pca.explained_variance_ratio_).round(3))

In [ ]:
# Scree plot
evr = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(evr) + 1), evr * 100, edgecolor='white')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_title('Scree Plot')

axes[1].plot(range(1, len(evr) + 1), np.cumsum(evr) * 100, 'bo-', linewidth=2, markersize=5)
axes[1].axhline(80, color='red', linestyle='--', label='80%')
axes[1].axhline(90, color='orange', linestyle='--', label='90%')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance (%)')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# PC1 vs PC2 coloured by popularity label
pca2 = PCA(n_components=2)
X_pca2 = pca2.fit_transform(X_scaled)

df['PC1'] = X_pca2[:, 0]
df['PC2'] = X_pca2[:, 1]

fig = px.scatter(df, x='PC1', y='PC2', color='popularity_label',
                 title='PCA: PC1 vs PC2 (coloured by Popularity Label)',
                 opacity=0.5)
fig.show()

In [ ]:
# PC1 vs PC2 coloured by energy category
fig = px.scatter(df, x='PC1', y='PC2', color='energy_category',
                 title='PCA: PC1 vs PC2 (coloured by Energy Category)',
                 opacity=0.5)
fig.show()

In [ ]:
# Feature loadings
loadings = pd.DataFrame(pca2.components_.T, index=features, columns=['PC1', 'PC2'])
print(loadings.round(3))

In [ ]:
# PC1 loadings bar chart
loadings_sorted = loadings['PC1'].sort_values()

fig = plt.figure(figsize=(10, 6))
colors = ['tomato' if v < 0 else 'blue' for v in loadings_sorted]
plt.barh(loadings_sorted.index, loadings_sorted.values, color=colors, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Loading Coefficient')
plt.title('PC1 Feature Loadings')
plt.tight_layout()
plt.show()

# Data Clustering

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# Use top 4 PCs for clustering
pca4 = PCA(n_components=4)
X_pca4 = pca4.fit_transform(X_scaled)

# Elbow method
inertias = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_pca4)
    inertias.append(km.inertia_)

fig = plt.figure(figsize=(10, 5))
plt.plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=6)
plt.xlabel('Number of Clusters k')
plt.ylabel('Inertia (WCSS)')
plt.title('Elbow Method for Optimal k')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

# Silhouette scores
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca4)
    sil = silhouette_score(X_pca4, labels, sample_size=2000, random_state=42)
    silhouettes.append(sil)
    print(f'k={k}  silhouette={sil:.4f}')

best_k = list(K_range)[np.argmax(silhouettes)]
print(f'\nBest k = {best_k}')

In [ ]:
# Fit final KMeans model
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['cluster'] = km_final.fit_predict(X_pca4)

print(df['cluster'].value_counts())

In [ ]:
# Cluster scatter in PCA space
fig = px.scatter(df, x='PC1', y='PC2',
                 color=df['cluster'].astype(str),
                 title=f'K-Means Clusters (k={best_k}) in PC1-PC2 Space',
                 labels={'color': 'Cluster'},
                 opacity=0.5)
fig.show()

In [ ]:
# Cluster mean profiles
profile_cols = ['danceability', 'energy', 'valence', 'acousticness',
                'instrumentalness', 'tempo', 'popularity', 'mood_score']

df.groupby('cluster')[profile_cols].mean().round(3)

In [ ]:
# Popularity distribution by cluster
fig = px.histogram(df, x='popularity', color=df['cluster'].astype(str),
                   barmode='overlay', nbins=20,
                   title='Popularity Distribution by Cluster',
                   labels={'color': 'Cluster'})
fig.show()

In [ ]:
# Box plot: danceability by cluster
fig = px.box(df, x=df['cluster'].astype(str), y='danceability',
             title='Danceability by Cluster',
             labels={'x': 'Cluster'})
fig.show()

In [ ]:
# Scatter: danceability vs valence coloured by cluster
fig = plt.figure(figsize=(16, 12))

plt.scatter(x=df['danceability'],
            y=df['valence'],
            s=df['energy'] * 100,
            c=df['cluster'],
            cmap='Set1',
            alpha=0.4,
            edgecolors='w')

plt.colorbar(label='Cluster')
plt.xlabel('Danceability')
plt.ylabel('Valence')
plt.title('Danceability vs Valence coloured by Cluster (size = energy)', y=1.05)

In [ ]:
df.columns